# NullVector Progress Notebook
## major-changes-v2 Phase F-J Cookbook Verification
Purpose: verify that the in-place LangGraph cookbook replacement is source-clean, framework-backed, and executable with deprecation warnings treated as errors.


### Environment

- This notebook validates the tracked `cookbook/langgraph_rag_cookbook.ipynb`.
- Execution runs the cookbook through `scripts/run_progress_notebook.py` with `PYTHONWARNINGS=error::DeprecationWarning`.
- The cookbook itself is fixture-first by default, so this progress notebook does not require a local real PDF or a live OpenRouter key.


In [ ]:
# environment setup
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

warnings.filterwarnings("error", category=DeprecationWarning)

REPO_ROOT = Path.cwd()
ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "cookbook-rag-remediation"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "repo_root": str(REPO_ROOT),
    "artifact_root": str(ARTIFACT_ROOT),
})


In [ ]:
# imports
import json
from pathlib import Path


In [ ]:
# configuration
COOKBOOK_NOTEBOOK = REPO_ROOT / "cookbook" / "langgraph_rag_cookbook.ipynb"
EXECUTED_NOTEBOOK = ARTIFACT_ROOT / "langgraph_rag_cookbook.executed.ipynb"
RUN_ENV = dict(os.environ)
RUN_ENV["PYTHONWARNINGS"] = "error::DeprecationWarning"


In [ ]:
# execution
source_notebook = json.loads(COOKBOOK_NOTEBOOK.read_text(encoding="utf-8"))
code_cells = [cell for cell in source_notebook["cells"] if cell.get("cell_type") == "code"]
source_blob = "\n".join("".join(cell.get("source", [])) for cell in code_cells)

source_checks = {
    "has_saved_outputs": any(cell.get("outputs") for cell in code_cells),
    "has_saved_error_outputs": any(
        output.get("output_type") == "error"
        for cell in code_cells
        for output in cell.get("outputs", [])
    ),
    "uses_missing_helper_module": "nullvector.cookbook_support" in source_blob,
    "uses_gateway_service": "GatewayService" in source_blob,
    "uses_direct_litellm_completion": "litellm.completion" in source_blob,
}

subprocess.run(
    [
        sys.executable,
        "scripts/run_progress_notebook.py",
        "--notebook",
        str(COOKBOOK_NOTEBOOK),
        "--output",
        str(EXECUTED_NOTEBOOK),
    ],
    cwd=REPO_ROOT,
    env=RUN_ENV,
    check=True,
)

executed_notebook = json.loads(EXECUTED_NOTEBOOK.read_text(encoding="utf-8"))
inspect_cell = next(
    cell
    for cell in executed_notebook["cells"]
    if cell.get("cell_type") == "code"
    and "".join(cell.get("source", [])).startswith("# inspect results")
)
inspect_output = "".join(
    text
    for output in inspect_cell.get("outputs", [])
    if output.get("output_type") == "stream"
    for text in output.get("text", [])
)
cookbook_summary = json.loads(inspect_output)


In [ ]:
# inspect results
summary = {
    "source_checks": source_checks,
    "cookbook_summary": cookbook_summary,
}
print(json.dumps(summary, indent=2, ensure_ascii=True))


### Known Limitations

- This progress notebook validates the fixture-first cookbook path, not a local real-PDF run.
- Live text remains optional and will skip honestly when `OPENROUTER_API_KEY` is not configured.
- Visual enrichment in the cookbook stays attachment-only and noop-backed by design.
